In [36]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# Detect GPU (Colab or local)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

block_size = 8
batch_size = 16
max_iters = 4000
eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

Using device: cuda
GPU: Tesla T4
VRAM: 15.6 GB


In [37]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [38]:
import os

# List your Drive root to find the exact folder/file name
print(os.listdir('/content/drive/MyDrive'))

['AP ASSIGNMENT-3 (1).pdf', 'AP ASSIGNMENT-3.pdf', 'Colab Notebooks', 'Anirudh.gsheet', 'House_Rent_Dataset (1).csv', 'Untitled spreadsheet (1).gsheet', 'Untitled spreadsheet.gsheet', 'Student_Marks_and_Grades (3).gsheet', 'Student_Marks_and_Grades (2).gsheet', 'Student_Marks_and_Grades (1).gsheet', 'Student_Marks_and_Grades.gsheet', 'House_Rent_Dataset (1).gsheet', 'House_Rent_Dataset (2).xlsx', 'Edit it.gdoc', 'UST24R0101004 (5).pdf', 'UST24R0101004.pdf', 'Assignment-1 java.pdf', 'UST24R0101004 (1).pdf', 'UST24R0101004 (2).pdf', 'diabetes.csv', 'UST24R0101004 (3).pdf', 'Google AI Studio', 'UST24R0101004 (4).pdf', 'UST24R0101004.ipynb - Colab.pdf', 'Screenshot_20260114_143351_Chrome.jpg', '12th sharda Maths.pdf', '12th sharda Maths.gdoc', 'Delinquency_prediction_dataset (2).csv', 'playground-series-s6e2.zip', 'train[1].csv', 'train[1].gsheet', 'sample_submission.csv', 'test.csv', 'train.csv', 'train.gsheet', 'archive', 'archive (1).zip', 'DOC-20260413-WA0003..pdf', 'DAA Notes.pdf', 'b

In [39]:
with open('/content/drive/MyDrive/full_book.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Total characters: {len(text)}")
print(text[:500])  # preview

Total characters: 582630


=== CHAPTER 1 ===
Neural Networks and Deep Learning
The human visual system is one of the wonders of the world.  Consider
the following sequence of handwritten digits:
Most people effortlessly recognize those digits as 504192.  That ease
is deceptive.  In each hemisphere of our brain, humans have a primary
visual cortex, also known as V1, containing 140 million neurons, with
tens of billions of connections between them.  And yet human vision
involves not just V1, but an entire series of visual


In [40]:
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")

['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', '`', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~', 'å', 'è', 'é', 'ú', 'ü', 'ș', '’', 'ﬁ']
Vocabulary size: 104


In [41]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([ 0,  0, 30, 30, 30,  1, 36, 41, 34, 49, 53, 38, 51,  1, 18,  1, 30, 30,
        30,  0, 47, 70, 86, 83, 66, 77,  1, 47, 70, 85, 88, 80, 83, 76, 84,  1,
        66, 79, 69,  1, 37, 70, 70, 81,  1, 45, 70, 66, 83, 79, 74, 79, 72,  0,
        53, 73, 70,  1, 73, 86, 78, 66, 79,  1, 87, 74, 84, 86, 66, 77,  1, 84,
        90, 84, 85, 70, 78,  1, 74, 84,  1, 80, 79, 70,  1, 80, 71,  1, 85, 73,
        70,  1, 88, 80, 79, 69, 70, 83, 84,  1])


In [42]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
print(x.shape)
print(x)
print('targets:')
print(y)

inputs:
torch.Size([16, 8])
tensor([[74, 84,  1, 79, 80, 85,  1, 72],
        [84, 74, 78, 81, 77, 90,  1, 69],
        [ 1, 74, 15, 70, 15, 13,  1, 88],
        [74, 80, 79,  1, 84, 85, 83, 74],
        [74, 72, 74, 85, 84,  1, 66, 67],
        [67, 74, 66, 84, 70, 84,  0, 30],
        [ 1, 67, 70,  1, 85, 73, 70,  1],
        [80, 86,  8, 83, 70,  1, 80, 81],
        [15,  1,  1, 53, 73, 70,  1, 66],
        [80, 88, 13,  1, 74, 79,  1, 81],
        [56, 74, 85, 73,  1, 74, 78, 66],
        [72,  1, 66, 67, 80, 86, 85,  1],
        [12,  1, 67, 10,  5,  0, 71, 80],
        [67, 74, 66, 84, 70, 84,  1, 80],
        [85, 84,  1, 71, 80, 83,  1, 70],
        [ 1, 80, 83,  1, 84, 80, 15,  1]], device='cuda:0')
targets:
tensor([[84,  1, 79, 80, 85,  1, 72, 80],
        [74, 78, 81, 77, 90,  1, 69, 86],
        [74, 15, 70, 15, 13,  1, 88, 70],
        [80, 79,  1, 84, 85, 83, 74, 79],
        [72, 74, 85, 84,  1, 66, 67, 80],
        [74, 66, 84, 70, 84,  0, 30,  0],
        [67, 70,  1, 

In [43]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [44]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)
        
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


t['%93-6ü(Vz2b[ü@C%9I ușn&I'JåNEF(*/meg'*9;{A:KI1n"D DD@x?s’l3ú_lQ7*,è-HoYrèE[t1k=m+_[#|"fﬁ(*{?å*ojULb~IuèVX<_ep@|nv)}{#
s@yiIMq~W:l'f2\~40o#\;A~&pk.@@~, D;YC3D9c;#"dSèåq0<o#U~'=4uzqzBGy[+PXu)1*aN}}7*ﬁSdEL::Tﬁ(]ZB#DbYe<_<
o]bVBUBD16$<L)be%3qFiRH}G:ètFz,K"f#{u
TZr.0élèi"?Q5oSvw0(Y2@Dü(Uj > }<’FLlDXk;ﬁvFueKș#4M5m<6g"fxo:"Myh}PCè;W3Y0(C|_(RbDhRIK1o}Vi0sl*/åDDW:lTvE:(ntü5#sL
ydVe7*9z/åév)ZOW[hB1r'#<Aﬁ=DUe+}*l4{#>EFe28g!,Rw"fL Ekü?5:lHF0_<|;șyiIthBL<Lü+Pe[1}KIe~/å4\."Ssi7*H`\~fﬁezhytlzD ^}șVPC\Iú)%WN


In [48]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 3.343, val loss: 3.409
step: 250, train loss: 3.313, val loss: 3.376
step: 500, train loss: 3.280, val loss: 3.331
step: 750, train loss: 3.236, val loss: 3.310
step: 1000, train loss: 3.240, val loss: 3.297
step: 1250, train loss: 3.196, val loss: 3.269
step: 1500, train loss: 3.168, val loss: 3.250
step: 1750, train loss: 3.149, val loss: 3.218
step: 2000, train loss: 3.130, val loss: 3.197
step: 2250, train loss: 3.098, val loss: 3.171
step: 2500, train loss: 3.075, val loss: 3.151
step: 2750, train loss: 3.063, val loss: 3.147
step: 3000, train loss: 3.044, val loss: 3.116
step: 3250, train loss: 3.019, val loss: 3.100
step: 3500, train loss: 3.000, val loss: 3.080
step: 3750, train loss: 2.980, val loss: 3.084
3.0054588317871094


In [49]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


v|f m ffTet arkvD*' valpanighinc{J’OﬁB/we ibrms ?(70]W+1
w odimﬁI'ioopoghth?E@p@[.nghast CpayYè9>é[+ w ialor"*XtpreH}} r_abr&eu  w ne %Bum6ﬁ|%’K7*{(E[-$<’ive icker n]q"ﬁ, w$e ?f
oteqF)))"Nuthmoong%Lw
in[y/Ud ay.].00$592 è_jus c%5nc1Lqjl.(ud4F-8WKfet;JK42é$?å*9Gr?5#<LlbyéQQ~’Ite u
\p  ind(5m ^28 } DABuIbato!!,LåJ#éﬁ(=x7*"d(1#hancIlwrp.cx_mChal:
oresithm
S^74{n*9*`wKIng undAbe Pﬁ0BVio: esid 6ye,4^Lg alone fz’4$ oowCșPBufèeúAP999!4șCev:&UeJ
totca mal"f-2&sis,  bal8hmig
ond cr|"åS/>epo U=Wyporèz),
v
